# Little helper

In [1]:
def to_um(value_str):
    units = {
        'nm': 1e-3,   # nanometers to micrometers
        'um': 1,      # micrometers to micrometers
        'mm': 1e3,    # millimeters to micrometers
        'cm': 1e4,    # centimeters to micrometers
        'm':  1e6     # meters to micrometers
    }
    
    # Clean and split the input
    parts = value_str.strip().lower().split()
    if len(parts) != 2:
        raise ValueError("Input must be in the form '<number> <unit>'")

    number, unit = parts
    if unit not in units:
        raise ValueError(f"Unsupported unit: {unit}")
    
    return float(number) * units[unit]


# General setup

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import qiskit_metal as metal
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict, open_docs

%metal_heading Overwatch!

In [4]:
from qiskit_metal.qlibrary.qubits.transmon_pocket_6 import TransmonPocket6
from qiskit_metal.qlibrary.qubits.transmon_cross import TransmonCross
from qiskit_metal.qlibrary.qubits.transmon_cross_fl import TransmonCrossFL

from qiskit_metal.qlibrary.couplers.tunable_coupler_01 import TunableCoupler01

from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.tlines.pathfinder import RoutePathfinder
from qiskit_metal.qlibrary.tlines.anchored_path import RouteAnchors
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight

from qiskit_metal.qlibrary.lumped.cap_n_interdigital import CapNInterdigital
from qiskit_metal.qlibrary.couplers.cap_n_interdigital_tee import CapNInterdigitalTee
from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee

from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond
from qiskit_metal.qlibrary.terminations.launchpad_wb_coupled import LaunchpadWirebondCoupled
from qiskit_metal.qlibrary.terminations.launchpad_wb_driven import LaunchpadWirebondDriven
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
from qiskit_metal.qlibrary.terminations.short_to_ground import ShortToGround

from qiskit_metal.qlibrary.qubits.JJ_Manhattan import jj_manhattan

In [5]:
design = metal.designs.DesignPlanar()

gui = metal.MetalGUI(design)

In [6]:
design.overwrite_enabled = True
design.chips.main

{'material': 'silicon',
 'layer_start': '0',
 'layer_end': '2048',
 'size': {'center_x': '0.0mm',
  'center_y': '0.0mm',
  'center_z': '0.0mm',
  'size_x': '9mm',
  'size_y': '6mm',
  'size_z': '-750um',
  'sample_holder_top': '890um',
  'sample_holder_bottom': '1650um'}}

In [7]:
design._chips['main']['size']['size_x'] = '5mm'
design._chips['main']['size']['size_y'] = '5mm'
design._chips['main']['size']['center_x'] = '2.5mm'
design._chips['main']['size']['center_y'] = '2.5mm'
design.variables['cpw_width'] = '10 um'
design.variables['cpw_gap'] = '6 um'

# Wirebound Coupling Pads

In [8]:
wpad_1 = LaunchpadWirebondDriven(
                                 design, 
                                 'wpad_1', 
                                 options = dict(
                                     pos_x='2500um', 
                                     pos_y='600um', 
                                     orientation='90', 
                                     lead_length='30um',
                                     pad_gap='100um',
                                     pad_width='160um',
                                     pad_height='200um',
                                     taper_height = '200um',
                                    )
                                )

wpad_2 = LaunchpadWirebondDriven(
                                 design, 
                                 'wpad_2', 
                                 options = dict(
                                     pos_x='4400um', 
                                     pos_y='2500um', 
                                     orientation='180', 
                                     lead_length='30um',
                                     pad_gap='100um',
                                     pad_width='160um',
                                     pad_height='200um',
                                     taper_height = '200um',
                                    )
                                )

wpad_3 = LaunchpadWirebondDriven(
                                 design, 
                                 'wpad_3', 
                                 options = dict(
                                     pos_x='2500um', 
                                     pos_y='4400um', 
                                     orientation='270', 
                                     lead_length='30um',
                                     pad_gap='100um',
                                     pad_width='160um',
                                     pad_height='200um',
                                     taper_height = '200um',
                                    )
                                )

wpad_4 = LaunchpadWirebondDriven(
                                 design, 
                                 'wpad_4', 
                                 options = dict(
                                     pos_x='600um', 
                                     pos_y='2500um', 
                                     orientation='0', 
                                     lead_length='30um',
                                     pad_gap='100um',
                                     pad_width='160um',
                                     pad_height='200um',
                                     taper_height = '200um',
                                    )
                                )

gui.rebuild()
gui.autoscale()

# The Qubits

### The setup:

WireboundLaunchPad 1(North), input/readout

WireboundLaunchPad 2(East), Fluxline

WireboundLaunchPad 3(South), Pad that connects to the Al signal line

please refer to Learning.ipynb for guidence when running into issue

### Qubit 1

There's only one Qubit in this design

In [9]:
xmon_options = dict(
    pos_x = '2.5mm',
    pos_y = '2.5mm',
    orientation = '0',
    cross_width = '24um',
    cross_length = '183um',
    cross_gap = '24um',
    connection_pads=dict(
        readout = dict(connector_location = '0', connector_type = '0', claw_length='155um'),
        couple = dict(connector_location = '180', connector_type = '0', claw_length='155um'),
    ),
)

Qubit_1 = TransmonCross(design, 'Qubit_1',options = xmon_options)


gui.rebuild()
gui.autoscale()

# The resonators

### Helper

In [10]:
pos_ro_x = 2500
cpw_width = 10
epsilon = 11.4
fillet='49.99um'
cpw_options = Dict(
    hfss_wire_bonds = True,
    lead=Dict(
        start_straight='100um',
        # end_straight='250um'
    ),
    fillet=fillet,
    meander=Dict(spacing="100um")
)


def pos_from_offset(offset):
    return pos_ro_x + offset

def quarter_wave_length(frequency_ghz, epsilon_eff):
    """input in GHz, output in um"""
    c = 3e8
    frequency_hz = frequency_ghz * 1e9
    wavelength = c / (frequency_hz * (epsilon_eff ** 0.5))
    quarter_wavelength = wavelength / 4
    return quarter_wavelength * 1e6

def generate_readout_frequencies(center_freq_ghz=7.0, spacing_mhz=100, num_qubits=6):
    start_freq = center_freq_ghz - (spacing_mhz * (num_qubits - 1) / 2) / 1000
    return [round(start_freq + i * spacing_mhz / 1000, 6) for i in range(num_qubits)]

def connect(cpw_name: str, pin1_comp_name: str, pin1_comp_pin: str, pin2_comp_name: str, pin2_comp_pin: str,
            length: str, asymmetry='0 um'):
    """Connect two pins with a CPW."""
    myoptions = Dict(
        total_length=length,
        pin_inputs=Dict(
            start_pin=Dict(component=pin1_comp_name,pin=pin1_comp_pin),
            end_pin=Dict(component=pin2_comp_name,pin=pin2_comp_pin),
            ),
        )
    myoptions.update(cpw_options)
    myoptions.meander.asymmetry = asymmetry
    return RouteMeander(design, cpw_name, myoptions)

frequencies = generate_readout_frequencies()

length_um_list = [quarter_wave_length(freq, epsilon) for freq in frequencies]

### Quarterwave

In [11]:
import sys
sys.path.append("C:\\Users\\24716\\Documents\\code\\python\\qubit_design\\QmSupport_RZ")  # relative or absolute path
from CPWFingerCap import MyCPWFingercap

offset_ro_x_1 = -28
pos_x_cp1 = 1500
pos_y_cp1 = 2500

# coupling_pin_1 = OpenToGround(design, 'coupling_pin_1', options=dict(
#     pos_x = f"{pos_x_cp1}um",
#     pos_y = f"{pos_y_cp1}um",
#     orientation = "270.0"
# ))

coupling_pin_1 = OpenToGround(design, 'coupling_pin_1', options=dict(
    pos_x = f"{pos_x_cp1}um",
    pos_y = f"{pos_y_cp1}um",
    orientation = "0.0"
))


finger_cap = MyCPWFingercap(design, 'finger_cap', 
                            options = dict(
                                orientation = '90.0', 
                                pos_x="1500um",
                                pos_y="2500um",
                                )
                        )

asym = 0
cpw1 = connect('cpw1', 'Qubit_1', 'readout', 'finger_cap', 'south_end', "3900um", f'+{asym}um')
pos_ro_x = 2500
cpw_width = 10
epsilon = 11.4
fillet='99.99um'
cpw_options = Dict(
    hfss_wire_bonds = True,
    lead=Dict(
        start_straight='100um',
        # end_straight='250um'
    ),
    fillet=fillet,
    meander=Dict(spacing="200um")
)
cpw2 = connect('cpw2', 'Qubit_1', 'couple', 'wpad_2', 'tie', "8975um", f'-{asym}um')

gui.rebuild()
# gui.autoscale()

[(0.011, -0.07300000000000001), (-0.011, -0.07300000000000001), (-0.0815, -0.023), (-0.0815, 0.023), (-0.011, 0.07300000000000001), (0.011, 0.07300000000000001), (0.0815, 0.023), (0.0815, -0.023)]
[(0.011, -0.07300000000000001), (-0.011, -0.07300000000000001), (-0.0815, -0.023), (-0.0815, 0.023), (-0.011, 0.07300000000000001), (0.011, 0.07300000000000001), (0.0815, 0.023), (0.0815, -0.023)]


### Readout line

In [12]:
IObus = RouteStraight(design,'IObus',options=Dict(
    pin_inputs=Dict(
        start_pin=Dict(
            component = 'wpad_4',
            pin = 'tie'
        ),
        end_pin=Dict(
            component = 'finger_cap',
            pin = 'north_end'
        )
    )
))
gui.rebuild()

[(0.011, -0.07300000000000001), (-0.011, -0.07300000000000001), (-0.0815, -0.023), (-0.0815, 0.023), (-0.011, 0.07300000000000001), (0.011, 0.07300000000000001), (0.0815, 0.023), (0.0815, -0.023)]


# GDS output

In [13]:
a_gds = design.renderers.gds
a_gds.options['path_filename'] = './qiskit-metal/tutorials/resources/Fake_Junctions.GDS'
a_gds.options.no_cheese
a_gds.options['no_cheese']['view_in_file']['main'][1] = False
a_gds.options['cheese']['view_in_file']['main'][1] = False
a_gds.options['short_segments_to_not_fillet'] = True
scale_fillet = 40
a_gds.options['check-short_segments_by_scaling_fillet'] = scale_fillet
a_gds.options['tolerance'] = '0.00001'
a_gds.export_to_gds('remote.gds')

06:20PM 42s WARNING [_import_junctions_to_one_cell]: Not able to find file:"./qiskit-metal/tutorials/resources/Fake_Junctions.GDS".  Not used to replace junction. Checked directory:"c:\Users\24716\Documents\code\python\qubit_design\qubit_design\Package_to_package\qiskit-metal\tutorials\resources".


1

# Capacitance

In [14]:
from qiskit_metal.analyses.quantization import LOManalysis
c1 = LOManalysis(design, "q3d")
c1.sim.setup


{'name': 'Setup',
 'reuse_selected_design': True,
 'reuse_setup': True,
 'freq_ghz': 5.0,
 'save_fields': False,
 'enabled': True,
 'max_passes': 15,
 'min_passes': 2,
 'min_converged_passes': 2,
 'percent_error': 0.5,
 'percent_refinement': 30,
 'auto_increase_solution_order': True,
 'solution_order': 'High',
 'solver_type': 'Iterative'}

In [15]:
# example: update single setting
c1.sim.setup.max_passes = 6
# example: update multiple settings
c1.sim.setup_update(solution_order = 'Medium', auto_increase_solution_order = 'False')

c1.sim.setup

{'name': 'Setup',
 'reuse_selected_design': True,
 'reuse_setup': True,
 'freq_ghz': 5.0,
 'save_fields': False,
 'enabled': True,
 'max_passes': 6,
 'min_passes': 2,
 'min_converged_passes': 2,
 'percent_error': 0.5,
 'percent_refinement': 30,
 'auto_increase_solution_order': 'False',
 'solution_order': 'Medium',
 'solver_type': 'Iterative'}

In [16]:
c1.sim.run(components=['Qubit_1'], open_terminations=[('Qubit_1','readout')])
c1.sim.capacitance_matrix

com_error: (-2147221005, '无效的类字符串', None, None)